# テーマ7 クラスタリング

## 目的
- クラスタリングが「正解ラベルなしで構造を探す」教師なし学習であることを理解する
- Wine データセットに k-means を適用し、似たデータをグループ化する
- PCA により高次元データを2次元で可視化する
- クラスタ番号に最初から意味があるわけではなく、後から特徴を見て意味づけすることを理解する

## 今日の成果物
- PCA の散布図を作る
- クラスタごとの特徴や真のラベルとの違いを短く説明する


## クラスタリングの基本

クラスタリングは、正解ラベルを使わずに、似ているデータ同士をグループ化する方法です。分類とは違い、最初から「正解のクラス」を教えません。

k-means では、各データ点を最も近いクラスタ中心に割り当て、中心を更新することを繰り返します。目的は、クラスタ内のばらつきを小さくすることです。

$$
\sum_{k=1}^{K}\sum_{x_i \in C_k} \|x_i - \mu_k\|^2
$$

ここで、$C_k$ はクラスタ、$\mu_k$ はクラスタ中心です。


### Step 0: ライブラリを読み込む
クラスタリングに必要なライブラリを読み込みます。標準化、k-means、PCA を使う準備を整えます。

### Step 0: クラスタリングの流れを確認する
このノートブックでは、Wine データを使って教師なし学習の基本を体験します。標準化、k-means、PCA による可視化を順番に試し、クラスタにどのような意味づけができるか考えてください。


In [ ]:
# ===== Step 0: ライブラリ読み込み =====
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.datasets import load_wine
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

sns.set_theme(style='whitegrid')

### Step 1: データを読み込む
Wine データセットを読み込み、クラスタリング対象のデータ構造を確認します。まずは先頭行と行列数を見て全体像を把握します.

In [ ]:
# ===== Step 1: データ読み込み =====
# TODO: WineデータセットをDataFrameにする
wine = load_wine(as_frame=True)
df = wine.frame.copy()

print('行数, 列数 =', df.shape)
display(df.head())

### Step 2: 標準化する
特徴量のスケールをそろえてからクラスタリングを行います。距離ベースの手法では、この前処理が結果に大きく影響します。

#### 解説: なぜ標準化するのか

k-means は距離にもとづいてグループを作ります。値のスケールが大きい特徴量があると、その特徴量だけが距離を強く支配してしまいます。

そのため、各特徴量を平均0、標準偏差1にそろえます。

$$
z = \frac{x - \mu}{\sigma}
$$

距離ベースの手法では、標準化は結果に大きく影響します。


In [ ]:
# ===== Step 2: 標準化 =====
# TODO: target を除いた特徴量だけを使う
feature_columns = wine.feature_names
X = df[feature_columns]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print('標準化後の形 =', X_scaled.shape)

### Step 3: k-means を実行する
クラスタ数を指定して、データを自動でグループ分けします。結果のラベルを見て、元の分類と比べてみてください。

#### 解説: クラスタ番号の注意

k-means が出す `0`, `1`, `2` という番号は、最初から意味を持っているわけではありません。例えば `cluster=0` が「高級ワイン」という意味だとは限りません。

意味づけをするには、各クラスタに含まれるデータの特徴をあとから確認します。


In [ ]:
# ===== Step 3: k-means =====
# TODO: クラスタ数を変えると結果がどう変わるか試す
n_clusters = 3
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_scaled)

df['cluster'] = cluster_labels
display(df[['target', 'cluster']].head())

In [ ]:
# ===== 追加演習: クラスタごとの平均値を見る =====
cluster_summary = df.groupby('cluster')[wine.feature_names].mean()
display(cluster_summary)


### Step 4: PCA で可視化する
2 次元に圧縮して、クラスタと真のラベルの見え方を比べます。教師なし学習の結果を視覚的に理解するための工程です。

#### 解説: PCAによる2次元可視化

Wine データには多くの特徴量があるため、そのままでは散布図にできません。PCA は、情報をできるだけ残しながら高次元データを少数の軸にまとめる方法です。

第1主成分は、データのばらつきが最も大きくなる方向です。第2主成分は、第1主成分と直交し、その次にばらつきが大きい方向です。

PCA の散布図は便利ですが、元の情報をすべて保っているわけではない点に注意します。


In [ ]:
# ===== Step 4: PCAで可視化 =====
# TODO: 2次元に圧縮して散布図を描く
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

plot_df = pd.DataFrame({
    'pca1': X_pca[:, 0],
    'pca2': X_pca[:, 1],
    'cluster': df['cluster'],
    'target': df['target'],
})

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.scatterplot(data=plot_df, x='pca1', y='pca2', hue='cluster', palette='Set2', ax=axes[0])
axes[0].set_title('k-means のクラスタ')

sns.scatterplot(data=plot_df, x='pca1', y='pca2', hue='target', palette='tab10', ax=axes[1])
axes[1].set_title('真のラベル')

plt.tight_layout()
plt.show()

In [ ]:
# ===== 追加演習: PCAがどれくらい情報を残しているか確認する =====
explained = pca.explained_variance_ratio_
print('第1主成分の寄与率:', explained[0])
print('第2主成分の寄与率:', explained[1])
print('2次元で説明できる割合:', explained.sum())


## 振り返り
- クラスタと真のラベルはどの程度似ていましたか。
- `n_clusters` を変えると、どんな違いがありましたか。
- クラスタ番号に意味をつけるには、何を確認する必要がありますか。
- 教師あり学習と教師なし学習の違いを自分の言葉で説明してください。
